<a href="https://colab.research.google.com/github/pattygirl665-lab/Wu_MA289_AY2701_Tree_Method_Housing_Competition/blob/main/Wu_MA289_AY2701_Tree_Method_Housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Creating the Submission File
Previous attempts before using the blended version include using a simple linear regression model using features of 'OverallQual' and 'GrLivArea,' including weighted features using 13 features weighted according to a seaborn heat map, using random forest regressor with an understanding of which 20 features were the most important along with correlation direction, one decisiontreeregressor using only the 'OverallQual' feature to how a single, highly important feature can predict 'SalePrice,' hyperparameterizing with GridSearchCV (searching for the best combination of max_depth, min_samples_split, and min_samples leaf) and using those best parameters to creae a better submission, trying hyperparameter tuning for randomforest with log transformation, creating polynomial features (basically creating a feature based on other features and weighting that) and retraining the tuned randomforest with these new forest, using XGBoost regressor with feature engineering and log transformation, then blending random forest and XGBoost predictions together. The blending one worked the best in hindsight. However, I did continue try making more feature-engineered data frames, removing outliers and abnormal data to clean the data, and tried to both blended models and XGBoost regressor or just RandomForest Regressor by itself. The blended one on data that wasn't cleaned was the best one I was able to get. The code for that one is below.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

#Define base path to my Google Drive
drive_folder_path = '/content/drive/MyDrive/Wu Colab Notebooks/Tree Method Housing'

#Data Loading
train_df = pd.read_csv(f'{drive_folder_path}/train.csv')
test_df = pd.read_csv(f'{drive_folder_path}/test.csv')
print('Train and Test Data loaded!')

#Fresh copies for processing to avoid modifying original loaded dataframes
train_df_extended_fe=train_df.copy()
test_df_extended_fe=test_df.copy()

#Transform SalePrice using Log (helps with scaling)
train_df_extended_fe['SalePrice'] = np.log1p(train_df_extended_fe['SalePrice'])

#Numerical Imputation and Feature Identification (converting the good/ex etc to 1-5s) for rest of code box
all_numeric_cols_base = train_df_extended_fe.select_dtypes(include=np.number).columns.drop(['Id','SalePrice'], errors='ignore').tolist()
common_numerical_features_base = list(set(all_numeric_cols_base) & set(test_df_extended_fe.select_dtypes(include=np.number).columns.drop('Id', errors='ignore').tolist()))

imputer_base = SimpleImputer(strategy='median')
train_df_extended_fe[common_numerical_features_base]=imputer_base.fit_transform(train_df_extended_fe[common_numerical_features_base])
test_df_extended_fe[common_numerical_features_base]=imputer_base.transform(test_df_extended_fe[common_numerical_features_base])

#Create existing interaction features
if 'OverallQual' in train_df_extended_fe.columns and 'GrLivArea' in train_df_extended_fe.columns:
  train_df_extended_fe['OverallQual_x_GrLivArea'] = train_df_extended_fe['OverallQual'] * train_df_extended_fe['GrLivArea']
  test_df_extended_fe['OverallQual_x_GrLivArea'] = test_df_extended_fe['OverallQual'] * test_df_extended_fe['GrLivArea']

if 'TotalBsmtSF' in train_df_extended_fe.columns and '1stFlrSF' in train_df_extended_fe.columns:
  train_df_extended_fe['TotalBsmtSF_x_1stFlrSF'] = train_df_extended_fe['TotalBsmtSF'] * train_df_extended_fe['1stFlrSF']
  test_df_extended_fe['TotalBsmtSF_x_1stFlrSF'] = test_df_extended_fe['TotalBsmtSF'] * test_df_extended_fe['1stFlrSF']

if 'GarageCars' in train_df_extended_fe.columns and 'GarageArea' in train_df_extended_fe.columns:
  train_df_extended_fe['GarageCars_x_GarageArea'] = train_df_extended_fe['GarageCars'] * train_df_extended_fe['GarageArea']
  test_df_extended_fe['GarageCars_x_GarageArea'] = test_df_extended_fe['GarageCars'] * test_df_extended_fe['GarageArea']

if 'YearBuilt' in train_df_extended_fe.columns and 'YearRemodAdd' in train_df_extended_fe.columns:
  train_df_extended_fe['YearBuilt_x_YearRemodAdd'] = train_df_extended_fe['YearBuilt'] * train_df_extended_fe['YearRemodAdd']
  test_df_extended_fe['YearBuilt_x_YearRemodAdd'] = test_df_extended_fe['YearBuilt'] * test_df_extended_fe['YearRemodAdd']

#Creating Existing Polynomial Features
poly_features_existing = ['GrLivArea', 'LotArea', 'TotalBsmtSF']
poly_transformer_existing = PolynomialFeatures(degree=2, include_bias=False)

for col in poly_features_existing:
  if col in train_df_extended_fe.columns:
      poly_train = poly_transformer_existing.fit_transform(train_df_extended_fe[[col]])
      poly_test = poly_transformer_existing.transform(test_df_extended_fe[[col]])
      poly_feature_names = [f'{col}_poly_{i+2}' for i in range(poly_train.shape[1]-1)]
      for i, name in enumerate(poly_feature_names):
        if name not in train_df_extended_fe.columns:
          train_df_extended_fe[name] = poly_train[:, i+1]
          test_df_extended_fe[name] = poly_test[:, i+1]

#Add new domain-specific features
train_df_extended_fe['TotalSF'] = train_df_extended_fe['GrLivArea'] + train_df_extended_fe['TotalBsmtSF']
test_df_extended_fe['TotalSF'] = test_df_extended_fe['GrLivArea'] + test_df_extended_fe['TotalBsmtSF']

train_df_extended_fe['TotalBathrooms'] = train_df_extended_fe['FullBath'] + 0.5* train_df_extended_fe['HalfBath'] + train_df_extended_fe['BsmtFullBath'] + 0.5 * train_df_extended_fe['BsmtHalfBath']
test_df_extended_fe['TotalBathrooms'] = test_df_extended_fe['FullBath'] + 0.5* test_df_extended_fe['HalfBath'] + test_df_extended_fe['BsmtFullBath'] + 0.5 * test_df_extended_fe['BsmtHalfBath']

train_df_extended_fe['HasPool'] = (train_df_extended_fe['PoolArea']>0).astype(int)
test_df_extended_fe['HasPool'] = (test_df_extended_fe['PoolArea']>0).astype(int)

train_df_extended_fe['HasFireplace'] = (train_df_extended_fe['Fireplaces']>0).astype(int)
test_df_extended_fe['HasFireplace'] = (test_df_extended_fe['Fireplaces']>0).astype(int)

train_df_extended_fe['HasGarage'] = (train_df_extended_fe['GarageArea']>0).astype(int)
test_df_extended_fe['HasGarage'] = (test_df_extended_fe['GarageArea']>0).astype(int)

train_df_extended_fe['YearsSinceBuilt'] = train_df_extended_fe['YrSold'] - train_df_extended_fe['YearBuilt']
test_df_extended_fe['YearsSinceBuilt'] = test_df_extended_fe['YrSold'] - test_df_extended_fe['YearBuilt']

train_df_extended_fe['YearsSinceRemodel'] = train_df_extended_fe['YrSold'] - train_df_extended_fe['YearRemodAdd']
test_df_extended_fe['YearsSinceRemodel'] = test_df_extended_fe['YrSold'] - test_df_extended_fe['YearRemodAdd']

#More features!
if 'OverallQual' in train_df_extended_fe.columns and 'TotalSF' in train_df_extended_fe.columns:
  train_df_extended_fe['OverallQual_x_TotalSF'] = train_df_extended_fe['OverallQual'] * train_df_extended_fe['TotalSF']
  test_df_extended_fe['OverallQual_x_TotalSF'] = test_df_extended_fe['OverallQual'] * test_df_extended_fe['TotalSF']

if 'OverallQual' in train_df_extended_fe.columns and 'YearBuilt' in train_df_extended_fe.columns:
  train_df_extended_fe['OverallQual_x_YearBuilt'] = train_df_extended_fe['OverallQual'] * train_df_extended_fe['YearBuilt']
  test_df_extended_fe['OverallQual_x_YearBuilt'] = test_df_extended_fe['OverallQual'] * test_df_extended_fe['YearBuilt']

if 'LotArea' in train_df_extended_fe.columns and 'OverallQual' in train_df_extended_fe.columns:
  train_df_extended_fe['LotArea_x_OverallQual'] = train_df_extended_fe['LotArea'] * train_df_extended_fe['OverallQual']
  test_df_extended_fe['LotArea_x_OverallQual'] = test_df_extended_fe['LotArea'] * test_df_extended_fe['OverallQual']

#Categorical Features (one-hot encoding)
categorical_cols = train_df_extended_fe.select_dtypes(include='object').columns.tolist()

for col in categorical_cols:
  train_df_extended_fe[col] = train_df_extended_fe[col].fillna('None')
  test_df_extended_fe[col] = test_df_extended_fe[col].fillna('None')

encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output=False)
encoded_train_features = encoder.fit_transform(train_df_extended_fe[categorical_cols])
encoded_train_df = pd.DataFrame(encoded_train_features, columns = encoder.get_feature_names_out(categorical_cols), index=train_df_extended_fe.index)

encoded_test_features = encoder.transform(test_df_extended_fe[categorical_cols])
encoded_test_df = pd.DataFrame(encoded_test_features, columns = encoder.get_feature_names_out(categorical_cols), index=test_df_extended_fe.index)

train_df_extended_fe = train_df_extended_fe.drop(columns=categorical_cols)
train_df_extended_fe = pd.concat([train_df_extended_fe, encoded_train_df], axis=1)

test_df_extended_fe = test_df_extended_fe.drop(columns=categorical_cols)
test_df_extended_fe = pd.concat([test_df_extended_fe, encoded_test_df], axis=1)

#Final Feature Alignment and type enforcement
X_train_extended_fe = train_df_extended_fe.drop(columns=['Id','SalePrice'], errors='ignore')
y_train_extended_fe = train_df_extended_fe['SalePrice']
X_test_extended_fe = test_df_extended_fe.drop(columns=['Id'], errors='ignore')

common_final_features = list(set(X_train_extended_fe.columns) & set(X_test_extended_fe.columns))
X_train_extended_fe = X_train_extended_fe[common_final_features].copy()
X_test_extended_fe = X_test_extended_fe[common_final_features].copy()

imputer_final_numeric = SimpleImputer(strategy='median')
for col in X_train_extended_fe.columns:
  if X_train_extended_fe[col].dtype == 'object':
    X_train_extended_fe[col] = pd.to_numeric(X_train_extended_fe[col], errors='coerce')
    X_test_extended_fe[col] = pd.to_numeric(X_test_extended_fe[col], errors='coerce')
    X_train_extended_fe[col] = imputer_final_numeric.fit_transform(X_train_extended_fe[[col]])
    X_test_extended_fe[col] = imputer_final_numeric.transform(X_test_extended_fe[[col]])

non_numeric_cols_train = X_train_extended_fe.select_dtypes(include='object').columns
if len(non_numeric_cols_train) > 0:
  X_train_extended_fe = X_train_extended_fe.drop(columns=non_numeric_cols_train)
  X_test_extended_fe = X_test_extended_fe.drop(columns=non_numeric_cols_train, errors = 'ignore')

print(f"Final feature count for training: {len(X_train_extended_fe.columns)}")

Train and Test Data loaded!
Final feature count for training: 320


Data processing above

In [ ]:
#Train Random Forest
#Used GridSearch in seperate code to find this/runtime takes a bit long if done
# Define the parameter grid to search
#param_grid_rf = {
   # 'n_estimators': [100, 200, 300], # Number of trees in the forest
   # 'max_depth': [10, 15, 20, None], # Maximum depth of the tree
   # 'min_samples_split': [2, 5, 10], # Minimum number of samples required to split an internal node
   # 'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
   # 'max_features': ['sqrt', 'log2'] # Number of features to consider when looking for the best split
#}
best_params_rf = {
    'n_estimators': 300,
    'max_depth': 15,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt'
}
best_rf_model = RandomForestRegressor(**best_params_rf, random_state=42)
best_rf_model.fit(X_train_extended_fe, y_train_extended_fe)

# Predict log(SalePrice)
best_rf_predictions_log = best_rf_model.predict(X_test_extended_fe)

# Inverse transform predictions back to original scale
rf_predictions_original = np.expm1(best_rf_predictions_log)
rf_predictions_original[rf_predictions_original < 0] = 0
print("RandomForest predictions generated.")

RandomForest predictions generated.


In [ ]:
#Train XGBoost ---
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', # For regression tasks
    n_estimators=1000,            # Number of boosting rounds
    learning_rate=0.05,           # Step size shrinkage
    max_depth=5,                  # Maximum depth of a tree
    subsample=0.7,                # Subsample ratio of the training instance
    colsample_bytree=0.7,         # Subsample ratio of columns when constructing each tree
    random_state=42,              # For reproducibility
    n_jobs=-1                     # Use all available CPU cores
)
xgb_model.fit(X_train_extended_fe, y_train_extended_fe)

# Predict log(SalePrice)
xgb_predictions_log = xgb_model.predict(X_test_extended_fe)

# Inverse transform predictions back to original scale
xgb_predictions_original = np.expm1(xgb_predictions_log)
xgb_predictions_original[xgb_predictions_original < 0] = 0
print("XGBoost predictions generated.")

XGBoost predictions generated.


In [ ]:
#Blend Predictions (Simple Average) ---
blended_predictions = (rf_predictions_original + xgb_predictions_original) / 2
blended_predictions[blended_predictions < 0] = 0

# Create submission DataFrame
draft_submission_blended_df = pd.DataFrame({
    'ID': test_df_extended_fe['Id'],
    'SalePrice': blended_predictions
})

# Save to Google Drive
submission_file_path_blended = f'{drive_folder_path}/draft_submission_blended.csv'
draft_submission_blended_df.to_csv(submission_file_path_blended, index=False)

print(f"\nBlended (RandomForest + XGBoost) submission file saved successfully to: {submission_file_path_blended}")
print("Preview of final submission:")
display(draft_submission_blended_df.head())


Blended (RandomForest + XGBoost) submission file saved successfully to: /content/drive/MyDrive/Wu Colab Notebooks/Tree Method Housing/draft_submission_blended.csv
Preview of final submission:


,ID,SalePrice
0,1461,122203.420847
1,1462,156427.457266
2,1463,183845.203514
3,1464,189688.637902
4,1465,186968.541069
